In [0]:
# Databricks notebook source

import dlt
from pyspark.sql.functions import (
    col, to_date, initcap, round, trim, lower,
    concat, substring, lit, regexp_extract, sha2
)

# COMMON FUNCTION

def read_bronze_stream(table_name):
    return spark.readStream.format("delta").table(f"retail_store_sales.bronze.{table_name}")


# MASKING FUNCTIONS (PII)

def mask_email(email_col):
    return concat(
        substring(email_col, 1, 3),
        lit("***@"),
        regexp_extract(email_col, "@(.+)", 1)
    )

def mask_phone(phone_col):
    return concat(
        substring(phone_col, 1, 3),
        lit("-XXX-XXX-"),
        substring(phone_col, -4, 4)
    )


# CUSTOMERS SILVER (WITH SK)


@dlt.table(
    name="customers_silver",
    comment="Cleaned, standardized, masked customers",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_drop("valid_customer_id", "customer_id IS NOT NULL")
@dlt.expect("valid_email", "email IS NOT NULL")
def customers_silver():

    df = read_bronze_stream("customers_bronze")

    return (
        df.dropDuplicates(["customer_id", "ingestion_ts"])
        .withColumn(
            "customer_sk",
            sha2(col("customer_id"), 256)
        )
        .select(
            col("customer_sk"),
            col("customer_id"),
            col("customer_name"),
            lower(trim(col("email"))).alias("email"),
            mask_email(col("email")).alias("email_masked"),
            col("phone"),
            mask_phone(col("phone")).alias("phone_masked"),
            initcap(trim(col("city"))).alias("city"),
            col("loyalty_tier"),
            col("ingestion_ts")
        )
    )


# SCD TYPE 2 - CUSTOMERS

dlt.create_streaming_table(
    name="customers_scd",
    comment="Customer SCD Type 2 history",
    table_properties={"quality": "silver"}
)

dlt.apply_changes(
    target="customers_scd",
    source="customers_silver",
    keys=["customer_id"],
    sequence_by=col("ingestion_ts"),
    stored_as_scd_type="2"
)


# PRODUCTS SILVER (WITH SK)

@dlt.table(
    name="products_silver",
    comment="Cleaned products",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_drop("valid_product_id", "product_id IS NOT NULL")
@dlt.expect("valid_price", "price > 0")
def products_silver():

    df = read_bronze_stream("products_bronze")

    return (
        df.dropDuplicates(["product_id", "ingestion_ts"])
        .withColumn(
            "product_sk",
            sha2(col("product_id"), 256)
        )
        .select(
            col("product_sk"),
            col("product_id"),
            initcap(trim(col("product_name"))).alias("product_name"),
            initcap(trim(col("category"))).alias("category"),
            round(col("price").cast("double"), 2).alias("price"),
            col("ingestion_ts")
        )
    )


# SCD TYPE 2 - PRODUCTS

dlt.create_streaming_table(
    name="products_scd",
    comment="Product SCD Type 2 history",
    table_properties={"quality": "silver"}
)

dlt.apply_changes(
    target="products_scd",
    source="products_silver",
    keys=["product_id"],
    sequence_by=col("ingestion_ts"),
    stored_as_scd_type="2"
)


# STORES SILVER (WITH SK)

@dlt.table(
    name="stores_silver",
    comment="Cleaned stores",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_drop("valid_store_id", "store_id IS NOT NULL")
def stores_silver():

    df = read_bronze_stream("stores_bronze")

    return (
        df.dropDuplicates(["store_id", "ingestion_ts"])
        .withColumn(
            "store_sk",
            sha2(col("store_id"), 256)
        )
        .select(
            col("store_sk"),
            col("store_id"),
            initcap(trim(col("store_name"))).alias("store_name"),
            initcap(trim(col("city"))).alias("city"),
            col("ingestion_ts")
        )
    )


# SCD TYPE 2 - STORES

dlt.create_streaming_table(
    name="stores_scd",
    comment="Store SCD Type 2 history",
    table_properties={"quality": "silver"}
)

dlt.apply_changes(
    target="stores_scd",
    source="stores_silver",
    keys=["store_id"],
    sequence_by=col("ingestion_ts"),
    stored_as_scd_type="2"
)

# SALES SILVER (NO JOINS)

@dlt.table(
    name="sales_silver",
    comment="Cleaned and validated sales data",
    table_properties={"quality": "silver"}
)
@dlt.expect_or_drop("valid_order_id", "order_id IS NOT NULL")
@dlt.expect("valid_customer", "customer_id IS NOT NULL")
@dlt.expect("valid_quantity", "quantity > 0")
@dlt.expect("valid_price", "price_per_unit > 0")
def sales_silver():

    df = read_bronze_stream("sales_bronze")

    return (
        df.dropDuplicates(["order_id"])
        .withColumn("quantity", col("quantity").cast("int"))
        .withColumn("price_per_unit", col("price_per_unit").cast("double"))
        .withColumn("discount_applied", col("discount_applied").cast("double"))
        .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))

        .withColumn("category", initcap(trim(col("category"))))
        .withColumn("payment_method", initcap(trim(col("payment_method"))))

        .withColumn(
            "total_amount",
            round(col("quantity") * col("price_per_unit") * (1 - col("discount_applied")), 2)
        )

        .withColumn(
            "data_quality_flag",
            (col("quantity") <= 0) |
            (col("price_per_unit") <= 0) |
            col("customer_id").isNull()
        )

        .filter((col("discount_applied") >= 0) & (col("discount_applied") <= 1))
    )


# INVENTORY EVENTS SILVER

@dlt.table(
    name="inventory_events_silver",
    comment="Cleaned inventory events",
    table_properties={"quality": "silver"}
)
@dlt.expect("valid_product", "product_id IS NOT NULL")
@dlt.expect("valid_store", "store_id IS NOT NULL")
@dlt.expect("valid_stock", "stock_level >= 0")
def inventory_events_silver():

    df = read_bronze_stream("inventory_events_bronze")

    return (
        df.select(
            col("product_id"),
            col("store_id"),
            col("stock_level").cast("int"),
            col("event_time"),
            col("ingestion_ts")
        )
    )